In [1]:
from src.res.gp.util.generator import gpGenerator
gen = gpGenerator()

In [2]:
fac = gen('ts_zscore(CP, 20)')

In [3]:
label = gen.gp.input.labels_raw

In [ ]:
from src.func import tensor as T
import torch
torch._logging.set_logs(dynamo=20, guards=True, recompiles=True)

assert fac.value is not None
x = fac.value
y = label

compiled_ic = torch.compile(T.rankic_2d)
compiled_ic_dyn = torch.compile(T.rankic_2d , dynamic = True)
x_cpu = x.cpu()
y_cpu = y.cpu()

def ic_1(x , y):
    ic = T.rankic_2d(x , y)
    return ic

def ic_2(x , y):
    ic = compiled_ic(x , y)
    return ic

def ic_3(x , y):
    ic = compiled_ic_dyn(x , y)
    return ic


In [45]:
x = torch.rand(5000,5000)

y = torch.rand(5000,5000)
ic_2(x,y)

tensor([ 0.0012, -0.0175,  0.0228,  ..., -0.0193, -0.0025, -0.0129])

In [46]:
%timeit ic_1(x , y)

5.84 s ± 130 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [47]:
%timeit ic_2(x , y)

5.36 s ± 99.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [49]:
%timeit ic_3(x , y)

5.28 s ± 37.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [51]:
torch.__version__

'2.7.1'

In [50]:
x = x.to('mps')
y = y.to('mps')
%timeit ic_1(x,y)

1.01 s ± 11.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
from src.proj import CALENDAR
from src.data import DATAVENDOR
from src.res.gp.util.input import InputElement , init_labels_raw , init_neutral_exp
from src.func import tensor as T
nday = 10
delay = 1
start_dt = 20220101 # gen.gp.input.start_dt
end_dt = 20231231 # gen.gp.input.end_dt
neutra = init_neutral_exp(start_dt , end_dt)
label = init_labels_raw(start_dt , end_dt , neutral_exp=neutra)

neutra , label


(tensor([[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.1888],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.1933],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.2034],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.0979],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.1090],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.1164]],
 
         [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.1888],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.1933],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.2034],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.0979],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.1090],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  1.1164]],
 
         [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -2.6085],
          [ 0.0000,  0.0000,

In [2]:
from src.data import DataBlock

block = DataBlock.load_raw('trade_ts' , 'day' , 20070101 , 20991231)
block = block.adjust_price().align_feature(['close'])

In [ ]:
import src.func.tensor as T
pctchg = T.pctchg(block.values , 5).roll(-5,1) # future 5 days rtn

In [4]:
weekly_pctchg = pctchg[:,::5].squeeze()

In [5]:
import torch
pctchg_rank = torch.floor(T.rank_pct(weekly_pctchg) / 0.05).clip(0,19) 

In [6]:
top_group = (pctchg_rank - 18).clip(0,1)[:,:-1]

cum_weeks = torch.cumsum(top_group.isfinite() , dim = 1)
cum_top_ratio = torch.cumsum(top_group.nan_to_num(0) , dim = 1) / cum_weeks
roll_top_ratio = torch.nn.functional.pad(top_group.unfold(1,50,1).nanmean(-1) , (49 , 0) , value = torch.nan)
next_top_chance = torch.nn.functional.pad(top_group.unfold(1,4,1).nansum(-1) , (0 , 3) , value = torch.nan)


In [77]:
cum_top_ratio.nanmean(1)

tensor([0.0358, 0.0503, 0.0385,  ..., 0.2039, 0.0268, 0.0698])

In [10]:
import pandas as pd
import numpy as np
date = block.date[::5][:-1]
d = {
    'secid' : np.repeat(block.secid , len(date)) ,
    'date' : np.tile(date , len(block.secid)) ,
    'cum_weeks' : torch.cumsum(top_group.isfinite() , dim = 1).flatten() ,
    'cum_top_ratio' : (torch.cumsum(top_group.nan_to_num(0) , dim = 1) / torch.cumsum(top_group.isfinite() , dim = 1)).flatten() ,
    'roll_top_ratio': torch.nn.functional.pad(top_group.unfold(1,50,1).nanmean(-1) , (49 , 0) , value = torch.nan).flatten() ,
    'next_top_chance' : torch.nn.functional.pad(top_group.unfold(1,4,1).nansum(-1) , (0 , 3) , value = torch.nan).flatten() 
}
[print(k , v.shape) for k, v in d.items()]
df = pd.DataFrame(d)
df = df.dropna(how = 'all')
df

secid (5016312,)
date (5016312,)
cum_weeks torch.Size([5016312])
cum_top_ratio torch.Size([5016312])
roll_top_ratio torch.Size([5016312])
next_top_chance torch.Size([5016312])


,secid,date,cum_weeks,cum_top_ratio,roll_top_ratio,next_top_chance
0,1,20070104,1,0.000000,NaN,0.0
1,1,20070111,2,0.000000,NaN,0.0
2,1,20070118,3,0.000000,NaN,0.0
3,1,20070125,4,0.000000,NaN,0.0
4,1,20070201,5,0.000000,NaN,0.0
...,...,...,...,...,...,...
5016307,920992,20250307,116,0.094828,0.10,1.0
5016308,920992,20250314,117,0.094017,0.10,1.0
5016309,920992,20250321,118,0.093220,0.10,NaN
5016310,920992,20250328,119,0.092437,0.10,NaN


In [68]:
cum_top_ratio[cum_weeks[:,-1] > 50][:,-1].nan_to_num(0).max()

tensor(0.2237)

In [69]:
cum_top_ratio[cum_weeks[:,-1] > 50][:,-1].nan_to_num(1).min()

tensor(0.)

In [7]:
block.values[0]

tensor([[[2.5015e+01, 3.6647e+02, 3.8323e+02,  ..., 3.5567e+00,
          4.9105e+00, 6.1647e+00]],

        [[2.5015e+01, 3.4846e+02, 3.4846e+02,  ..., 2.7614e+00,
          3.8126e+00, 4.7863e+00]],

        [[2.5015e+01, 3.2645e+02, 3.3870e+02,  ..., 1.6323e+00,
          2.2536e+00, 2.8292e+00]],

        ...,

        [[1.2778e+02, 1.3967e+03, 1.4069e+03,  ..., 4.3431e-01,
          4.3432e-01, 9.7992e-01]],

        [[1.2778e+02, 1.3992e+03, 1.4158e+03,  ..., 4.2457e-01,
          4.2458e-01, 9.5794e-01]],

        [[1.2778e+02, 1.4107e+03, 1.4299e+03,  ..., 3.7468e-01,
          3.7469e-01, 8.4537e-01]]])

In [ ]:
from src.api.summary import SummaryAPI
SummaryAPI.update()

*************************************************** Summary Start at 2026-02-06 00:16:20 ***************************************************
Summary of Account Period Return:


gru_avg                       gru_day                      df_scores_v0                     ht_master_combined                    
                t50       scr       rev       t50       scr       rev        t50         scr       rev           t50            scr       rev   
StartDate     20170104  20170104  20170104  20170104  20170104  20170104   20191218    20191218  20191218      20180102       20180102  20180102
EndDate       20250107  20250107  20250107  20250107  20250107  20250107   20241219    20241219  20241219      20241219       20241219  20241219
Total            8.222     9.697     9.122     8.149     8.549     8.725      4.663       2.594     4.145        10.017          6.495    13.378
Annualized       0.319     0.344     0.335     0.318     0.325     0.328      0.413       0.291     0.387         0.411          0.335     0.466
Y2017            0.469     0.545     0.526     0.457     0.503     0.429        NaN         NaN       NaN           NaN            NaN       NaN
Y2018            0.397     0.588     0.548     0.293     0.612     0.428        NaN         NaN       NaN         0.435          0.667     0.882
Y2019            0.437     0.488     0.549     0.611     0.426     0.612      0.011        0.02     0.032         0.811          0.509     0.739
Y2020            0.649      0.44     0.396     0.737     0.481     0.584       0.54       0.666     0.633          0.63          0.512     0.593
Y2021             0.19     0.285     0.295      0.27     0.285     0.339      0.881       0.374     0.607         0.581          0.319     0.477
Y2022            0.173     0.126     0.106     0.074     0.113       0.1      0.212       0.189     0.277         0.086           0.07     0.102
Y2023            0.125     0.141     0.137     0.032      0.09     0.069       0.31       0.169     0.215          0.17          0.113     0.162
Y2024            0.268     0.306     0.289     0.297     0.281     0.258      0.218       0.107     0.225         0.294          0.256     0.458
Y2025           -0.047    -0.057    -0.055     -0.05    -0.065    -0.059        NaN         NaN       NaN           NaN            NaN       NaN
Last Week       -0.047    -0.057    -0.055     -0.05    -0.065    -0.059     -0.024      -0.024    -0.027        -0.027         -0.026    -0.024
Last Month      -0.038    -0.077     -0.08     -0.06      -0.1    -0.083      0.063      -0.008     0.055         0.054         -0.003     0.065
Last Quarter    -0.037    -0.005    -0.022     0.016    -0.047    -0.026       0.32       0.187     0.295         0.376          0.188     0.405
Last Year         0.21     0.224     0.213     0.233      0.19       0.2      0.227       0.109     0.231         0.287          0.268     0.456

******************************************** Summary Finish at 2026-02-06 00:16:21, Cost <1 Sec ********************************************
